In [20]:
import pandas as pd
import numpy as np

df = pd.read_csv("resources/loan_data.csv")

print(df)

mean_income = 100000

        ID  Gender      Race  Age Age_Group  Income  Credit_Score  \
0      415  Female     White   62   Over 60   90470           790   
1      334    Male     White   49     25-60  106317           615   
2     4658  Female  Hispanic   58     25-60   96596           698   
3     1794  Female     White   41     25-60   69501           840   
4     9101    Male     White   32     25-60  128322           779   
...    ...     ...       ...  ...       ...     ...           ...   
7495   474    Male     White   55     25-60   63231           581   
7496   840  Female     White   62   Over 60  110949           544   
7497  3355  Female     Black   48     25-60  132120           843   
7498  7727  Female  Hispanic   68   Over 60  166576           610   
7499  6906  Female     White   39     25-60   58622           582   

      Loan_Amount Employment_Type Education_Level Citizenship_Status  \
0          100269       Full-time      Bachelor's            Citizen   
1          278731       Ful

In [21]:
def assign_criminal_record(row):
    # 1. Start with a baseline probability (e.g., 5%)
    prob = 0.05
    
    # 2. Race-based likelihood (Reflecting US DOJ statistics)
    race_multipliers = {
        'Black': 2.5,    # Statistically higher contact with justice system
        'Hispanic': 1.5,
        'White': 0.8,
        'Asian': 0.5,
        'Multiracial': 1.2
    }
    prob *= race_multipliers.get(row['Race'], 1.0)
    
    # 3. Education-based likelihood (Inverse correlation)
    edu_multipliers = {
        'High School': 1.5,
        'Some College': 1.0,
        'Bachelor\'s': 0.5,
        'Graduate': 0.2
    }
    prob *= edu_multipliers.get(row['Education_Level'], 1.0)
    
    # 4. Gender Factor (Men are statistically more likely to have records)
    if row['Gender'] == 'Male':
        prob *= 2.0

    # Ensure probability doesn't exceed 100%
    prob = np.clip(prob, 0, 1)
    
    return np.random.choice(['Yes', 'No'], p=[prob, 1 - prob])

# Apply this FIRST
df['Criminal_Record'] = df.apply(assign_criminal_record, axis=1)

In [ ]:
def assign_education_by_race(race):
    # Education Levels
    levels = ['High School', 'Some College', "Bachelor's", 'Graduate']
    
    # Probabilities based on Race (approximating US Census trends)
    if race == 'White':
        probs = [0.10, 0.15, 0.40, 0.35]
    elif race == 'Asian':
        probs = [0.28, 0.30, 0.25, 0.17]
    elif race == 'Black':
        probs = [0.40, 0.35, 0.18, 0.07]
    elif race == 'Hispanic':
        probs = [0.45, 0.30, 0.17, 0.08]
    else: # Multiracial / Other
        probs = [0.30, 0.35, 0.25, 0.10]
        
    return np.random.choice(levels, p=probs)

# Step 1: Generate Education based on Race
df['Education_Level'] = df['Race'].apply(assign_education_by_race)

In [23]:



import numpy as np

def calculate_realistic_income(row):
    # 1. Set Base Income by Zip Code Group
    if row['Zip_Code_Group'] == 'Urban Professional':
        income = np.random.normal(95000, 15000)
    elif row['Zip_Code_Group'] == 'High-income Suburban':
        income = np.random.normal(110000, 20000)
    elif row['Zip_Code_Group'] == 'Working Class Urban':
        income = np.random.normal(45000, 8000)
    else: # Historically Redlined / Other
        income = np.random.normal(38000, 7000)


    # Education Multipliers (Real-world "Value of a Degree")
    multipliers = {
        'High School': 1.0,
        'Some College': 1.2,
        'Bachelor\'s': 1.8,
        'Graduate': 2.5
    }
    
    income = income * multipliers[row['Education_Level']]
    
    income += np.random.normal(0, 2000)

    if row['Criminal_Record'] == 'Yes':
        # Apply a 30% reduction with a bit of randomness
        penalty = np.random.uniform(0.25, 0.40) 
        income = income * (1 - penalty)

    if row['Language_Proficiency'] == 'Limited':
        # Apply a 30% reduction with a bit of randomness
        penalty = np.random.uniform(0.25, 0.40) 
        income = income * (1 - penalty)

    return round(income, 2)



# Apply to your dataframe
df['Income'] = df.apply(calculate_realistic_income, axis=1)


In [24]:
def generate_credit_score(row):
    # 1. Start with a "Fair" baseline
    score = 650 
    
    # 2. Age Modifier (Credit history length)
    if row['Age_Group'] == 'Over 60':
        score += 70
    elif row['Age_Group'] == '25-60':
        score += 20
    else: # Under 25
        score -= 50

    # 3. Income Modifier
    if row['Income'] > 80000:
        score += 50
    elif row['Income'] < 35000:
        score -= 60

    # 4. Criminal Record Modifier (Reflecting gaps in credit history)
    if row['Criminal_Record'] == 'Yes':
        score -= 40

    # 5. Add "Noise" (Real life isn't perfectly predictable)
    score += np.random.normal(0, 30)

    # Clip the score to the standard 300-850 range
    return int(np.clip(score, 300, 850))

df['Credit_Score'] = df.apply(generate_credit_score, axis=1)


In [25]:
def determine_loan_status(row):
    # 1. Start with a base probability of 70%
    approval_prob = 0.70

    score = row['Credit_Score']
    
    if score >= 740:     # Excellent
        approval_prob = 0.95
    elif score >= 670:   # Good
        approval_prob = 0.80
    elif score >= 580:   # Fair
        approval_prob = 0.40
    else:                # Poor / Very Poor
        approval_prob = 0.10
    
    # 2. Income Impact (The most realistic factor)
    if row['Income'] < 35000:
        approval_prob -= 0.40  # Massive penalty for low income
    elif row['Income'] < 55000:
        approval_prob -= 0.15  # Moderate penalty
    elif row['Income'] > 100000:
        approval_prob += 0.20  # High-income boost

    # 3. Direct Criminal Record Impact
    # Banks often see a record as a "risk multiplier" regardless of income
    if row['Criminal_Record'] == 'Yes':
        approval_prob -= 0.25

    

    # Ensure probability stays between 0 and 1
    approval_prob = max(0, min(1, approval_prob))
    
    return np.random.choice(['Approved', 'Denied'], p=[approval_prob, 1 - approval_prob])

# Update your dataframe
df['Loan_Approved'] = df.apply(determine_loan_status, axis=1)

In [26]:
df.to_csv("resources/new_loan.csv")

In [27]:
df


,ID,Gender,Race,Age,Age_Group,Income,Credit_Score,Loan_Amount,Employment_Type,Education_Level,Citizenship_Status,Language_Proficiency,Disability_Status,Criminal_Record,Zip_Code_Group,Loan_Approved
0,415,Female,White,62,Over 60,176718.65,797,100269,Full-time,Bachelor's,Citizen,Fluent,No,No,Urban Professional,Approved
1,334,Male,White,49,25-60,112291.36,728,278731,Full-time,Bachelor's,Citizen,Fluent,No,No,Urban Professional,Approved
2,4658,Female,Hispanic,58,25-60,125001.50,711,414326,Full-time,Bachelor's,Citizen,Fluent,Yes,No,Urban Professional,Approved
3,1794,Female,White,41,25-60,142671.38,743,414801,Part-time,Bachelor's,Citizen,Fluent,No,No,Urban Professional,Approved
4,9101,Male,White,32,25-60,194261.50,751,298557,Full-time,Bachelor's,Citizen,Fluent,No,No,High-income Suburban,Approved
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7495,474,Male,White,55,25-60,73102.60,694,88918,Full-time,Bachelor's,Citizen,Fluent,No,No,Historically Redlined,Denied
7496,840,Female,White,62,Over 60,101810.67,772,186714,Full-time,Graduate,Citizen,Fluent,No,No,Historically Redlined,Approved
7497,3355,Female,Black,48,25-60,71380.61,695,260626,Full-time,Bachelor's,Citizen,Fluent,Yes,No,Working Class Urban,Approved
7498,7727,Female,Hispanic,68,Over 60,81907.83,817,430139,Full-time,Some College,Citizen,Fluent,Yes,No,Urban Professional,Denied
